# Verbessertes Modell 2 – Random Forest

**Ziel:** Ein zweites Modell mit einem anderen Algorithmus bauen und mit Modell 1 (Logistische Regression, F1 = 0,55) vergleichen.

## 1. Daten laden und vorbereiten

- **Ziel (y):** `bank_account`, umgewandelt in 1 (Konto) und 0 (kein Konto).
- **Merkmale (X):** alle Spalten außer `uniqueid` (nur eine Nummer), `year` (durch das Land schon bestimmt) und dem Ziel selbst.
- Damit nutzen wir dieselben Merkmale wie die Baseline und Modell 1, sodass der Vergleich fair ist.

In [2]:
import pandas as pd  # Werkzeugkasten pandas holen  # Kap. 15

df = pd.read_csv("data/raw/Train.csv")  # Trainingsdaten laden  # Kap. 15

y = df["bank_account"].map({"Yes": 1, "No": 0})  # Ziel: Yes wird 1, No wird 0  # neu
X = df.drop(columns=["uniqueid", "year", "bank_account"])  # Merkmale ohne ID, Jahr und Ziel  # Kap. 17

print(X.shape)  # Größe der Merkmalstabelle  # Kap. 15
print(y.mean())  # Anteil „Yes“  # Kap. 15

(23524, 10)
0.14079238224791701


In [3]:
from sklearn.compose import ColumnTransformer  # Werkzeug: Spalten unterschiedlich behandeln  # neu
from sklearn.preprocessing import OneHotEncoder  # Werkzeug: Text in 0/1-Spalten umwandeln  # neu
from sklearn.pipeline import Pipeline  # Werkzeug: Schritte hintereinander schalten  # neu
from sklearn.ensemble import RandomForestClassifier  # Werkzeug: Random Forest  # neu
from sklearn.model_selection import StratifiedKFold  # Werkzeug: faire Aufteilung in 5 Teile  # neu
from sklearn.model_selection import cross_val_score  # Werkzeug: Score pro Teil berechnen  # neu

text_spalten = X.select_dtypes(exclude="number").columns  # alle Text-Spalten finden  # neu
zahl_spalten = X.select_dtypes(include="number").columns  # alle Zahlen-Spalten finden  # neu

vorbereitung = ColumnTransformer([  # Vorbereitung der Spalten  # neu
    ("text", OneHotEncoder(handle_unknown="ignore"), text_spalten),  # Text wird zu 0/1-Spalten  # neu
    ("zahl", "passthrough", zahl_spalten),  # Zahlen bleiben, wie sie sind  # neu
])  # Ende der Vorbereitung  # neu

wald = RandomForestClassifier(  # Random Forest anlegen  # neu
    n_estimators=200,  # 200 Entscheidungsbäume  # neu
    class_weight="balanced",  # seltene Klasse „Konto“ stärker gewichten  # neu
    random_state=42,  # fester Startwert, gleiches Ergebnis bei jedem Lauf  # neu
    n_jobs=-1,  # alle Prozessorkerne nutzen, damit es schneller geht  # neu
)  # Ende Random Forest  # neu

modell = Pipeline([  # Vorbereitung und Modell verbinden  # neu
    ("vorbereitung", vorbereitung),  # Schritt 1: Spalten vorbereiten  # neu
    ("modell", wald),  # Schritt 2: Random Forest  # neu
])  # Ende der Pipeline  # neu

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 5 gemischte Teile mit je 14 % „Yes“  # neu
f1_werte = cross_val_score(modell, X, y, cv=cv, scoring="f1")  # F1 in jedem der 5 Durchgänge  # neu

print("F1 pro Durchgang:", f1_werte.round(3))  # die 5 einzelnen Werte  # Kap. 14
print("F1 Durchschnitt:", f1_werte.mean().round(3))  # Mittelwert der 5 Werte  # Kap. 14

F1 pro Durchgang: [0.453 0.476 0.479 0.447 0.454]
F1 Durchschnitt: 0.462


## 2. Random Forest mit 5-facher Cross-Validation

**Was ist ein Random Forest?** Viele Entscheidungsbäume (hier 200), die jeweils auf einem zufälligen Teil der Daten lernen. Am Ende stimmen alle Bäume ab. Anders als die Logistische Regression erkennt er auch Kombinationen von Merkmalen selbst (z. B. Handy × Bildung).

**Aufbau (Pipeline):**
1. Text-Spalten werden per One-Hot-Encoding in 0/1-Spalten umgewandelt.
2. Zahlen-Spalten (Alter, Haushaltsgröße) bleiben unverändert. Ein Random Forest braucht keine Standardisierung.
3. `class_weight="balanced"` gleicht das Ungleichgewicht (nur 14,1 % mit Konto) aus.

**Prüfung:** 5-fache stratifizierte Cross-Validation (gemischt, `random_state=42`), genau wie bei Modell 1.

**Ergebnis:** F1 = 0,462 (die 5 Durchgänge liegen zwischen 0,447 und 0,479).

Der Random Forest mit Standard-Einstellungen ist damit **schlechter** als die Baseline (0,494) und Modell 1 (0,550). Vermutliche Ursache: Die Bäume wachsen ohne Grenze sehr tief und lernen die Trainingsdaten auswendig (Overfitting). Im nächsten Schritt begrenzen wir die Bäume (Tuning).

In [4]:
from sklearn.model_selection import GridSearchCV  # Werkzeug: alle Einstellungen durchprobieren  # neu

raster = {  # welche Einstellungen getestet werden  # Kap. 10
    "modell__max_depth": [10, 20, None],  # maximale Tiefe der Bäume (None = ohne Grenze)  # Kap. 8
    "modell__min_samples_leaf": [1, 5, 10],  # Mindestzahl Personen pro Blatt  # Kap. 8
}  # Ende des Rasters  # Kap. 10

suche = GridSearchCV(modell, raster, cv=cv, scoring="f1")  # Suche mit 5-facher CV anlegen  # neu
suche.fit(X, y)  # alle 9 Kombinationen trainieren und prüfen  # neu

print("Beste Einstellung:", suche.best_params_)  # Gewinner-Kombination  # neu
print("Bester F1:", round(suche.best_score_, 3))  # F1 der besten Kombination  # Kap. 4

Beste Einstellung: {'modell__max_depth': 20, 'modell__min_samples_leaf': 5}
Bester F1: 0.513


## 3. Tuning der Bäume (GridSearchCV)

Wir probieren 9 Kombinationen aus zwei Einstellungen aus, jeweils mit 5-facher Cross-Validation:

- **max_depth** (maximale Tiefe): 10, 20 oder ohne Grenze. Flachere Bäume lernen weniger auswendig.
- **min_samples_leaf** (Mindestgröße eines Blatts): 1, 5 oder 10 Personen. Größere Blätter machen das Modell robuster.

Gewählt wird die Kombination mit dem höchsten F1-Score.

**Ergebnis:** Beste Einstellung: `max_depth = 20`, `min_samples_leaf = 5`. F1 = 0,513 (5-fache CV).

Das Tuning verbessert den Random Forest deutlich (von 0,462 auf 0,513). Begrenzte Bäume lernen weniger auswendig und erkennen allgemeinere Muster.

In [5]:
import numpy as np  # Werkzeugkasten NumPy holen  # Kap. 14
from sklearn.model_selection import cross_val_predict  # Werkzeug: Vorhersagen aus der CV holen  # neu
from sklearn.metrics import f1_score  # F1 berechnen  # neu
from sklearn.metrics import precision_score  # Precision berechnen  # neu
from sklearn.metrics import recall_score  # Recall berechnen  # neu

bestes_modell = suche.best_estimator_  # das beste Modell aus dem Tuning  # neu
wahrsch = cross_val_predict(bestes_modell, X, y, cv=cv, method="predict_proba")  # Wahrscheinlichkeiten für alle Personen  # neu
wahrsch_ja = wahrsch[:, 1]  # nur die Spalte „Konto“  # Kap. 14

schwellen = np.arange(0.3, 0.81, 0.05).round(2)  # Grenzen 0.3, 0.35 ... 0.8  # Kap. 14

beste_schwelle = 0.5  # Startwert  # Kap. 4
bester_f1 = 0  # Startwert  # Kap. 4
for s in schwellen:  # alle Grenzen durchgehen  # Kap. 7
    f1 = f1_score(y, (wahrsch_ja >= s).astype(int))  # F1 bei dieser Grenze  # neu
    if f1 > bester_f1:  # besser als bisher?  # Kap. 6
        bester_f1 = f1  # neuen Bestwert merken  # Kap. 4
        beste_schwelle = s  # passende Grenze merken  # Kap. 4

vorhersage_best = (wahrsch_ja >= beste_schwelle).astype(int)  # Vorhersage mit der besten Grenze  # neu

print("F1 bei Schwelle 0.5:", round(f1_score(y, (wahrsch_ja >= 0.5).astype(int)), 3))  # Vergleichswert  # neu
print("Beste Schwelle:", beste_schwelle)  # gewählte Grenze  # Kap. 4
print("F1 (Yes):", round(f1_score(y, vorhersage_best), 3))  # Endergebnis Modell 2  # neu
print("Precision (Yes):", round(precision_score(y, vorhersage_best), 3))  # Trefferquote der Ja-Meldungen  # neu
print("Recall (Yes):", round(recall_score(y, vorhersage_best), 3))  # gefundene Kontoinhaber  # neu

F1 bei Schwelle 0.5: 0.513
Beste Schwelle: 0.65
F1 (Yes): 0.549
Precision (Yes): 0.513
Recall (Yes): 0.589


## 4. Angepasste Entscheidungsschwelle

Normalerweise sagt das Modell ab 50 % Wahrscheinlichkeit „Konto“. Wie bei Modell 1 testen wir Grenzen von 0,30 bis 0,80 und wählen die mit dem höchsten F1-Score.

**Einschränkung:** Die Schwelle wird auf denselben CV-Vorhersagen gewählt, auf denen gemessen wird. Das Ergebnis ist daher leicht optimistisch und wird in Milestone 3 auf einem separaten Testteil geprüft.

**Ergebnis:**

| Schwelle | F1 (Yes) | Precision (Yes) | Recall (Yes) |
| --- | --- | --- | --- |
| 0,50 (Standard) | 0,513 | – | – |
| **0,65 (beste)** | **0,549** | **0,513** | **0,589** |

Mit der Schwelle 0,65 erreicht der Random Forest F1 = 0,549 und liegt damit praktisch gleichauf mit Modell 1 (0,550). Er findet etwas mehr Kontoinhaber (Recall 0,589 statt 0,566), macht dafür aber etwas mehr Fehlalarme (Precision 0,513 statt 0,535).

## Fazit: Verbessertes Modell 2

| Stufe | F1 (Yes) |
| --- | --- |
| Random Forest, Standard-Einstellungen | 0,462 |
| + Tuning (max_depth = 20, min_samples_leaf = 5) | 0,513 |
| **+ Schwelle 0,65 statt 0,5** | **0,549** |

**Endergebnis Modell 2:** F1 = 0,549 · Precision = 0,513 · Recall = 0,589 (5-fache CV)

**Vergleich mit Modell 1 (Logistische Regression):**

| Modell | F1 | Precision | Recall |
| --- | --- | --- | --- |
| Modell 1: Logistische Regression | 0,550 | 0,535 | 0,566 |
| Modell 2: Random Forest | 0,549 | 0,513 | 0,589 |

**Einordnung:** Beide Modelle sind praktisch gleich gut. Der Random Forest findet etwas mehr Kontoinhaber, macht aber mehr Fehlalarme. Die Logistische Regression ist einfacher und besser zu erklären. Wie bei Modell 1 hat die angepasste Schwelle den größten Effekt.

**Einschränkung:** Die Schwelle wurde auf denselben CV-Vorhersagen gewählt. Welches Modell wirklich besser ist, prüfen wir in Milestone 3 auf einem separaten Testteil.